# Medical RAG — 배치 실험 데이터 수집
7개 질환(감기·천식·독감·고혈압·당뇨병·코로나·결핵) 30개 한글 질문을 자동 실행하여  
RAGAS 평가 결과를 Supabase `rag_audit_log` 테이블에 적재합니다.  
답변은 출력하지 않으며, 시각화 분석용 실험 데이터 생성이 목적입니다.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath("."))   # 프로젝트 루트를 모듈 경로에 추가

from graph import run_medical_self_corrective_rag
print("imports OK")

imports OK


In [ ]:
# 7개 질환 × 각 4~5개 = 총 30개 한글 질문
QUESTIONS = [
    # ── 감기 (4) ──────────────────────────────────────────────
    ("감기", "감기에 걸렸을 때 가장 효과적인 치료 방법은 무엇인가요?"),
    ("감기", "감기 증상이 일주일 이상 지속되면 어떻게 해야 하나요?"),
    ("감기", "어린이 감기에 항생제를 사용해야 하나요?"),
    ("감기", "감기 예방을 위해 어떤 생활 습관이 중요한가요?"),

    # ── 천식 (4) ──────────────────────────────────────────────
    ("천식", "천식 발작이 발생했을 때 즉각적인 응급 처치 방법은 무엇인가요?"),
    ("천식", "천식 환자가 피해야 할 환경적 요인에는 어떤 것들이 있나요?"),
    ("천식", "소아 천식의 주요 증상과 진단 방법은 무엇인가요?"),
    ("천식", "천식과 COPD는 어떻게 감별하나요?"),

    # ── 독감 (4) ──────────────────────────────────────────────
    ("독감", "독감 백신 접종은 언제 맞는 것이 가장 효과적인가요?"),
    ("독감", "타미플루(오셀타미비르) 처방 기준과 복용 방법은 무엇인가요?"),
    ("독감", "독감 합병증으로 어떤 질환이 발생할 수 있으며 고위험군은 누구인가요?"),
    ("독감", "독감과 코로나19 증상을 임상적으로 어떻게 구별하나요?"),

    # ── 고혈압 (5) ────────────────────────────────────────────
    ("고혈압", "고혈압 진단 기준과 정상 혈압 범위는 어떻게 되나요?"),
    ("고혈압", "고혈압 약은 평생 복용해야 하나요, 중단할 수 있나요?"),
    ("고혈압", "고혈압 환자에게 권장되는 DASH 식이요법이란 무엇인가요?"),
    ("고혈압", "고혈압성 응급증(hypertensive emergency) 증상과 치료는 무엇인가요?"),
    ("고혈압", "임신 중 고혈압(전자간증)은 어떻게 관리하나요?"),

    # ── 당뇨병 (5) ────────────────────────────────────────────
    ("당뇨병", "제1형 당뇨병과 제2형 당뇨병의 차이점은 무엇인가요?"),
    ("당뇨병", "당뇨병 환자의 혈당 관리 목표(HbA1c 기준)는 어떻게 되나요?"),
    ("당뇨병", "인슐린 자가 주사 방법과 주의사항을 알려주세요."),
    ("당뇨병", "당뇨병의 주요 만성 합병증 종류와 예방법은 무엇인가요?"),
    ("당뇨병", "저혈당 발생 시 응급 대처 방법은 무엇인가요?"),

    # ── 코로나19 (4) ──────────────────────────────────────────
    ("코로나", "코로나19 치료에 사용되는 항바이러스제 종류와 적응증은 무엇인가요?"),
    ("코로나", "코로나19 후유증(Long COVID)의 주요 증상과 관리 방법은 무엇인가요?"),
    ("코로나", "코로나19 고위험군 환자의 입원 기준은 어떻게 되나요?"),
    ("코로나", "코로나19 감염 후 격리 기간과 해제 기준은 무엇인가요?"),

    # ── 결핵 (4) ──────────────────────────────────────────────
    ("결핵", "결핵 치료에 사용되는 1차 항결핵제(HRZE)의 종류와 복용 기간은 얼마나 되나요?"),
    ("결핵", "잠복 결핵 감염과 활동성 결핵의 차이점 및 치료 방법은 무엇인가요?"),
    ("결핵", "다제내성 결핵(MDR-TB)이란 무엇이며 어떻게 치료하나요?"),
    ("결핵", "결핵 전파 경로와 효과적인 예방 방법은 무엇인가요?"),
]

print(f"총 {len(QUESTIONS)}개 질문 준비 완료")
for i, (disease, q) in enumerate(QUESTIONS, 1):
    print(f"  {i:2d}. [{disease}] {q}")

In [ ]:
import time

TOTAL = len(QUESTIONS)
results = []   # (index, disease, question, request_id, success, elapsed_sec, error)

print(f"{'─'*70}")
print(f"  배치 실행 시작 — 총 {TOTAL}개 질문 (답변 출력 없음, Supabase만 적재)")
print(f"{'─'*70}\n")

for i, (disease, question) in enumerate(QUESTIONS, 1):
    t0 = time.perf_counter()
    success = False
    request_id = ""
    error_msg = ""

    try:
        # 기존 RAG 파이프라인 호출 — step_callback=None 으로 UI 출력 없이 실행
        # save_audit_log()는 _critic_node() 내부에서 자동 호출됨
        final_state = run_medical_self_corrective_rag(
            question,
            forced_user_level=None,   # 자동 분류
            step_callback=None,       # 화면 출력 없음
            llm_provider="openai",
        )
        request_id = final_state.get("request_id", "")
        success = True

    except Exception as e:
        error_msg = str(e)

    elapsed = time.perf_counter() - t0
    results.append((i, disease, question, request_id, success, elapsed, error_msg))

    status = "OK" if success else "FAIL"
    rid_short = request_id[:8] + "..." if request_id else "—"
    print(f"[{i:2d}/{TOTAL}] {status}  [{disease}]  {elapsed:.1f}s  rid={rid_short}")
    if error_msg:
        print(f"         오류: {error_msg}")

print(f"\n{'─'*70}")
ok_count  = sum(1 for r in results if r[4])
fail_count = TOTAL - ok_count
print(f"  완료: 성공 {ok_count}개 / 실패 {fail_count}개")
print(f"  → Supabase rag_audit_log 테이블에 {ok_count}건 이상 적재됨")
print(f"{'─'*70}")

In [ ]:
# 적재 결과 검증 — Supabase에서 방금 생성된 request_id들의 로그 행 수 확인
import psycopg2
import config.settings as settings

ok_request_ids = [r[3] for r in results if r[4] and r[3]]

if ok_request_ids:
    conn = psycopg2.connect(settings.SUPABASE_DB_URL)
    cur = conn.cursor()
    placeholders = ",".join(["%s::uuid"] * len(ok_request_ids))
    cur.execute(
        f"""
        SELECT
            COUNT(*)                                      AS total_rows,
            COUNT(DISTINCT request_id)                    AS distinct_requests,
            AVG(ragas_f)                                  AS avg_f,
            AVG(ragas_ar)                                 AS avg_ar,
            AVG(ragas_cp)                                 AS avg_cp,
            SUM(CASE WHEN is_escalated THEN 1 ELSE 0 END) AS escalated_count,
            SUM(CASE WHEN is_fallback  THEN 1 ELSE 0 END) AS fallback_count
        FROM public.rag_audit_log
        WHERE request_id IN ({placeholders})
        """,
        ok_request_ids,
    )
    row = cur.fetchone()
    conn.close()

    print("─── Supabase 적재 검증 ───────────────────────────────────")
    print(f"  총 로그 행 수       : {row[0]}")
    print(f"  고유 request_id 수  : {row[1]}")
    print(f"  평균 Faithfulness   : {row[2]:.3f}" if row[2] else "  평균 Faithfulness   : —")
    print(f"  평균 Answer Rel.    : {row[3]:.3f}" if row[3] else "  평균 Answer Rel.    : —")
    print(f"  평균 Context Prec.  : {row[4]:.3f}" if row[4] else "  평균 Context Prec.  : —")
    print(f"  에스컬레이션 발생   : {row[5]}건")
    print(f"  Fallback 발생       : {row[6]}건")
    print("──────────────────────────────────────────────────────────")
else:
    print("성공한 요청이 없어 검증을 건너뜁니다.")